[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/07-interview-prep/01_sql.ipynb)

*Interview Prep · Notebook 1 of 5*

# SQL for Interviews

The SQL that interviews ask: joins, GROUP BY and HAVING, subqueries, CTEs and window functions, including second-highest salary, top N per group, running totals and finding gaps.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** SQL describes the result you want, and the database works out how. Read a query in the order it runs, and most of the mystery goes.

**What interviewers really test here:** Joins and what they do to row counts, NULLs, GROUP BY vs window functions, and top-N-per-group.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. How to read a query: the order SQL really runs in *(Basics)*
- [ ] 2. Joins, and how they multiply rows *(Basics)*
- [ ] 3. Subqueries and the second-highest salary *(Core)*
- [ ] 4. Window functions: the interview favourite *(Core)*
- [ ] 5. Hard patterns: gaps, islands, pivots and hierarchies *(Advanced)*
- [ ] 6. Indexes, performance and design *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline
import math
import numpy as np
import matplotlib.pyplot as plt
PLAIN = True

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Picking the tool

| Question sounds like | Reach for |
| --- | --- |
| "for each group, one total" | GROUP BY |
| "groups where…" | HAVING |
| "has no…", "never…" | LEFT JOIN … IS NULL |
| "top N per…" | ROW\_NUMBER, RANK or DENSE\_RANK in a CTE |
| "running…", "cumulative…" | SUM() OVER |
| "compared with the previous…" | LAG |
| "consecutive…" | value − ROW\_NUMBER |
| "percent of total" | SUM(x) OVER () |
| "reports to", "tree" | recursive CTE |

### NULL rules

- `x = NULL` is never true: use `IS NULL`.
- `COUNT(*)` counts rows, `COUNT(col)` skips NULLs.
- SUM and AVG ignore NULLs, so AVG divides by non-NULL rows only.
- `COALESCE(x, 0)` gives a default.
- `NOT IN` with a NULL in the list returns nothing.

---
# Part 2: Lessons, step by step

## Lesson 1: How to read a query: the order SQL really runs in

*Level: Basics*

SQL is written in one order and runs in another. Most beginner confusion comes from here:



| Step | Clause | What it does |
| --- | --- | --- |
| 1 | FROM / JOIN | gather the rows |
| 2 | WHERE | keep rows |
| 3 | GROUP BY | make piles |
| 4 | HAVING | keep piles |
| 5 | SELECT | choose and compute columns |
| 6 | ORDER BY | sort |
| 7 | LIMIT | cut off |



This explains two classic errors. You can't use `WHERE COUNT(*) > 2`, because counts don't exist yet at step 2, so that's what HAVING is for. And in standard SQL, a column alias from SELECT isn't visible in WHERE, because SELECT happens later.



**Worked example.** "Departments with at least 2 people, and an average salary above 80":



```python
SELECT dept_id, AVG(salary)
FROM employees
WHERE dept_id IS NOT NULL      -- step 2: drop Ivan, who has no department
GROUP BY dept_id               -- step 3: piles for 1, 2, 3
HAVING COUNT(*) >= 2 AND AVG(salary) > 80;   -- step 4: pile 3 has only Hana, so it goes
```



Pile 1 is 150, 120, 120, 90, with an average of 120. Pile 2 is 100, 70, 80, with an average of 83.33. Both survive.



**In the interview,** say the steps as you write. Interviewers want to see how you think, and a wrong column name won't sink you if your logic is clear.

**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q2](#q2), [Q3](#q3), [Q7](#q7). Try one now, or carry on and do them in Part 3.

## Lesson 2: Joins, and how they multiply rows

*Level: Basics*

A join matches rows from two tables. Picture two lists side by side, with lines drawn between matches.



| Join | Keeps | Unmatched columns |
| --- | --- | --- |
| INNER | only rows that match | none are kept |
| LEFT | all left rows | right side becomes NULL |
| RIGHT | all right rows | left side becomes NULL |
| FULL | all rows from both | NULL on the missing side |
| CROSS | every combination | n × m rows |



**Two important questions interviewers like:**



**1. Join two tables of 3 rows each. How many rows can result?** Anything from 0 to 9. If the key is duplicated, every match pairs up: a key appearing 2 times on the left and 3 on the right makes 6 rows. This is the classic cause of an inflated total, where revenue suddenly doubles after a join.



**2. A filter on a LEFT JOIN's right table.** `WHERE o.amount > 0` throws away the NULL rows, so your LEFT JOIN quietly becomes an INNER JOIN. To keep unmatched rows, put the condition in `ON` instead.



**Finding what's missing** is a very common question. LEFT JOIN, then keep rows where the right side is NULL. In our data, the Legal department has no employees, and customer Dev has no orders.



**NOT IN has a trap:** `x NOT IN (1, NULL)` is never true, because comparing with NULL gives "unknown". One NULL in the subquery and you get no rows at all.



**Self join:** a table joined to itself, using two aliases, such as employee and manager. In our data, Fay earns 70 and her manager Hana earns 60, so Fay out-earns her manager.

**✏️ Your turn:** you now have what you need for [Q4](#q4), [Q5](#q5), [Q8](#q8). Try one now, or carry on and do them in Part 3.

## Lesson 3: Subqueries and the second-highest salary

*Level: Core*

Probably the most-asked SQL question. Salaries are 150, 120, 120, 100, 90, 80, 70, 60, 50.



**Method 1, MAX below MAX:**



```python
SELECT MAX(salary) FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees);
```



The inner query gives 150, and the outer gives the highest below it: **120**. It's easy to understand, but it doesn't extend to the 5th highest.



**Method 2, OFFSET:**



```python
SELECT DISTINCT salary FROM employees
ORDER BY salary DESC LIMIT 1 OFFSET 1;
```



DISTINCT matters here. Without it, the second row is the other 120 by luck, and the third highest would wrongly also be 120.



**Method 3, DENSE\_RANK,** which is best for "Nth per group": next lesson.



**Follow-ups to expect:**



- *What if there's no second salary?* Method 1 returns NULL. Method 2 returns no rows, so wrap it in a subquery if you need a NULL.
- *Ties?* Ask whether "second highest" means distinct values. This kind of clarifying question scores points.



**Duplicates.** Finding them is GROUP BY plus HAVING COUNT(\*) &gt; 1. To keep one of each, use ROW\_NUMBER, again next.

**✏️ Your turn:** you now have what you need for [Q9](#q9), [Q6](#q6). Try one now, or carry on and do them in Part 3.

## Lesson 4: Window functions: the interview favourite

*Level: Core*

GROUP BY squashes each group into one row. A **window function** computes across related rows, but **keeps every row**.



```python
function() OVER (PARTITION BY group_column ORDER BY sort_column)
```



**The three rankings differ only on ties.** Salaries 150, 120, 120, 90:



| Name | Salary | ROW\_NUMBER | RANK | DENSE\_RANK |
| --- | --- | --- | --- | --- |
| Asha | 150 | 1 | 1 | 1 |
| Ben | 120 | 2 | 2 | 2 |
| Chen | 120 | 3 | 2 | 2 |
| Dara | 90 | 4 | **4** | **3** |



- **ROW\_NUMBER:** always unique, so use it to keep exactly one row per group.
- **RANK:** leaves a gap after a tie.
- **DENSE\_RANK:** no gap, so use it for "top 2 distinct salaries".



**You can't filter a window function in WHERE**, because it's computed after WHERE. Wrap it in a CTE:



```python
WITH r AS (
  SELECT name, dept, salary,
         DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dr
  FROM employees)
SELECT * FROM r WHERE dr <= 2;
```



**Others to know:**



- `SUM(x) OVER (ORDER BY d)`: a running total. For 100, 50, 200: 100, 150, 350.
- `LAG(x)`, `LEAD(x)`: the previous or next row. For monthly revenue 350, 100, 160, change is NULL, −250, +60.
- `AVG(x) OVER (PARTITION BY g)`: each row next to its group's average.
- `SUM(SUM(x)) OVER ()`: the grand total, inside a GROUP BY, for percentage of total.



**A trap the demo reveals:** Ben and Chen both show a running total of 390, not 270 and 390. With ties in ORDER BY, the default window treats equal rows as one group. To add row by row, order by something unique, or write `ROWS UNBOUNDED PRECEDING`.



Run the demo, then change PARTITION BY to see the ranks restart per department.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
import sqlite3
DEMO_SCHEMA = """
CREATE TABLE employees (name TEXT, dept TEXT, salary INTEGER);
INSERT INTO employees VALUES ('Asha','Eng',150), ('Ben','Eng',120), ('Chen','Eng',120), ('Dara','Eng',90), ('Eli','Sales',100), ('Fay','Sales',70);
"""
def run(query):
    con = sqlite3.connect(":memory:")
    con.executescript(DEMO_SCHEMA)
    return con.execute(query).fetchall()


def window_demo():
    rows = run("""
        SELECT dept, name, salary,
          ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS row_num,
          DENSE_RANK() OVER (PARTITION BY dept ORDER BY salary DESC) AS dense,
          SUM(salary) OVER (PARTITION BY dept ORDER BY salary DESC) AS running
        FROM employees ORDER BY dept, salary DESC""")
    for row in rows:
        print(row)


window_demo()

('Eng', 'Asha', 150, 1, 1, 150)
('Eng', 'Ben', 120, 2, 2, 390)
('Eng', 'Chen', 120, 3, 2, 390)
('Eng', 'Dara', 90, 4, 3, 480)
('Sales', 'Eli', 100, 1, 1, 100)
('Sales', 'Fay', 70, 2, 2, 170)


**✏️ Your turn:** you now have what you need for [Q10](#q10), [Q11](#q11), [Q12](#q12), [Q13](#q13), [Q17](#q17), [Q14](#q14), [Q18](#q18), [Q15](#q15). Try one now, or carry on and do them in Part 3.

## Lesson 5: Hard patterns: gaps, islands, pivots and hierarchies

*Level: Advanced*

**Gaps and islands: find a streak.** The trick looks like magic until you see it. User 1 logged in on days 1, 2, 3 and 5. Subtract each row's position:



| Day | ROW\_NUMBER | Day − row |
| --- | --- | --- |
| 1 | 1 | **0** |
| 2 | 2 | **0** |
| 3 | 3 | **0** |
| 5 | 4 | **1** |



Within a streak, day and row number rise together, so their difference stays constant. Group by that difference and count: streaks of 3 and 1, so the longest is **3**.



**Pivot with CASE.** Turn rows into columns, using conditional aggregation:



```python
SELECT customer_id,
  SUM(CASE WHEN month = 'Jan' THEN amount ELSE 0 END) AS jan,
  SUM(CASE WHEN month = 'Feb' THEN amount ELSE 0 END) AS feb
FROM orders GROUP BY customer_id;
```



**Recursive CTE: walk a hierarchy.** It has two parts: a starting row, and a step joining back to itself, which repeats until no new rows appear.



```python
WITH RECURSIVE tree(id, name, depth) AS (
    SELECT id, name, 0 FROM employees WHERE manager_id IS NULL
    UNION ALL
    SELECT e.id, e.name, t.depth + 1
    FROM employees e JOIN tree t ON e.manager_id = t.id)
SELECT * FROM tree;
```



Round 1 finds Asha at depth 0. Round 2 finds those reporting to her at depth 1. Round 3 finds their reports at depth 2. Then it stops.

**✏️ Your turn:** you now have what you need for [Q19](#q19), [Q16](#q16), [Q20](#q20). Try one now, or carry on and do them in Part 3.

## Lesson 6: Indexes, performance and design

*Level: Advanced*

Senior interviews go beyond writing queries.



**An index** is like the index at the back of a book. Usually it's a B-tree, which turns an O(n) scan into O(log n). With a billion rows, that's about 30 steps instead of a billion. The costs are extra storage and slower writes, because each index must be updated.



**A composite index on (a, b)** helps queries on a, or on a and b, but generally not on b alone. Think of a phone book sorted by surname, then first name: you can't quickly find everyone called "Sam".



**Things that stop an index being used:** a function on the column (`WHERE YEAR(d) = 2024`), a leading wildcard (`LIKE '%x'`), and low-selectivity columns.



**How to diagnose a slow query:** read the plan with `EXPLAIN`, look for full scans on big tables, add suitable indexes, and filter early.



| Term | Meaning |
| --- | --- |
| Normalisation | each fact stored once, avoiding update errors |
| Denormalisation | deliberate duplication, for faster reads |
| ACID | atomic, consistent, isolated, durable transactions |
| Primary key | uniquely identifies a row |
| Foreign key | a reference to a row in another table |



**Dialects differ.** Practice here uses SQLite. Window functions and CTEs are broadly the same elsewhere, but date functions and `LIMIT` vary. For example, SQL Server uses `TOP`. Ask which database the interviewer has in mind.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

**▶ Run this cell before the questions.** It sets up the data and tools they share.

In [3]:
import sqlite3

SCHEMA = """
CREATE TABLE departments (id INTEGER PRIMARY KEY, name TEXT);
INSERT INTO departments VALUES (1, 'Eng'), (2, 'Sales'), (3, 'HR'), (4, 'Legal');

CREATE TABLE employees (id INTEGER PRIMARY KEY, name TEXT, dept_id INTEGER, manager_id INTEGER, salary INTEGER);
INSERT INTO employees VALUES
 (1, 'Asha', 1, NULL, 150), (2, 'Ben', 1, 1, 120), (3, 'Chen', 1, 1, 120), (4, 'Dara', 1, 2, 90),
 (5, 'Eli', 2, 1, 100), (6, 'Fay', 2, 8, 70), (7, 'Gus', 2, 5, 80), (8, 'Hana', 3, 1, 60), (9, 'Ivan', NULL, 1, 50);

CREATE TABLE customers (id INTEGER PRIMARY KEY, name TEXT, city TEXT);
INSERT INTO customers VALUES (1, 'Ann', 'Pune'), (2, 'Bob', 'Delhi'), (3, 'Cid', 'Pune'), (4, 'Dev', 'Goa');

CREATE TABLE orders (id INTEGER PRIMARY KEY, customer_id INTEGER, amount INTEGER, order_date TEXT);
INSERT INTO orders VALUES
 (1, 1, 100, '2024-01-05'), (2, 1, 50, '2024-01-20'), (3, 2, 200, '2024-01-25'), (4, 1, 70, '2024-02-02'),
 (5, 3, 30, '2024-02-10'), (6, 2, 80, '2024-03-01'), (7, 2, 80, '2024-03-02');

CREATE TABLE logins (user_id INTEGER, day INTEGER);
INSERT INTO logins VALUES (1, 1), (1, 2), (1, 3), (1, 5), (2, 2), (2, 3), (3, 7);

CREATE TABLE emails (id INTEGER, email TEXT);
INSERT INTO emails VALUES (1, 'a@x.com'), (2, 'b@x.com'), (3, 'a@x.com'), (4, 'c@x.com'), (5, 'b@x.com');
"""

def run_sql(query):
    """Run a query on a fresh database and return the rows as lists."""
    con = sqlite3.connect(":memory:")
    con.executescript(SCHEMA)
    rows = con.execute(query).fetchall()
    con.close()
    return [list(r) for r in rows]


<a id="q1"></a>
### Q1. Filter and Sort

🟢 Warm-up · *WHERE, ORDER BY*

Return name and salary of employees earning more than 90, highest first, then by name.

**Examples**

- `` → `[['Asha', 150], ['Ben', 120], ['Chen', 120], ['Eli', 100]]`

In [ ]:
def high_earners():
    """
    Return name and salary of employees earning more than 90, highest first,
    then by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(high_earners, [
    ((), [['Asha', 150], ['Ben', 120], ['Chen', 120], ['Eli', 100]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Select everything, then filter and sort in Python. Works, but it ignores what the database is for.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`WHERE` and `ORDER BY` let the database do it.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Select everything, then filter and sort in Python. Works, but it ignores what the database is for.

```python
def high_earners():
    rows = run_sql("SELECT name, salary FROM employees")
    kept = [r for r in rows if r[1] > 90]
    return sorted(kept, key=lambda r: (-r[1], r[0]))
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `WHERE` and `ORDER BY` let the database do it.

```python
def high_earners():
    return run_sql("""
        SELECT name, salary FROM employees
        WHERE salary > 90
        ORDER BY salary DESC, name""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Select everything, then filter and sort in Python. Works, but it ignores what the database is for. That costs O(n log n) time and O(n) space."

"I can do better: `WHERE` and `ORDER BY` let the database do it. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/filter-and-sort)

<a id="q2"></a>
### Q2. Count per Group

🟢 Warm-up · *GROUP BY*

Return each dept\_id and its employee count, ignoring employees with no department, ordered by dept\_id.

**Examples**

- `` → `[[1, 4], [2, 3], [3, 1]]`

In [ ]:
def dept_counts():
    """
    Return each dept_id and its employee count, ignoring employees with no
    department, ordered by dept_id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(dept_counts, [
    ((), [[1, 4], [2, 3], [3, 1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A correlated subquery for each department.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`GROUP BY`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A correlated subquery for each department.

```python
def dept_counts():
    return run_sql("""
        SELECT id, (SELECT COUNT(*) FROM employees e WHERE e.dept_id = d.id) AS n
        FROM departments d
        WHERE n > 0
        ORDER BY id""")
```

**Cost:** O(d·n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `GROUP BY`.

```python
def dept_counts():
    return run_sql("""
        SELECT dept_id, COUNT(*) FROM employees
        WHERE dept_id IS NOT NULL
        GROUP BY dept_id
        ORDER BY dept_id""")
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A correlated subquery for each department. That costs O(d·n) time and O(d) space."

"I can do better: `GROUP BY`. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/count-per-group)

<a id="q3"></a>
### Q3. Groups Above a Limit

🟢 Warm-up · *HAVING*

Return dept\_id and average salary for departments with at least 2 employees and an average over 80, ordered by dept\_id.

**Examples**

- `` → `[[1, 120.0], [2, 83.3333]]`

In [ ]:
def big_departments():
    """
    Return dept_id and average salary for departments with at least 2 employees
    and an average over 80, ordered by dept_id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(big_departments, [
    ((), [[1, 120.0], [2, 83.3333]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Group in a subquery, then filter outside with WHERE.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`HAVING` filters groups directly.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Group in a subquery, then filter outside with WHERE.

```python
def big_departments():
    return run_sql("""
        SELECT dept_id, avg_s FROM (
            SELECT dept_id, AVG(salary) AS avg_s, COUNT(*) AS n
            FROM employees WHERE dept_id IS NOT NULL GROUP BY dept_id)
        WHERE n >= 2 AND avg_s > 80
        ORDER BY dept_id""")
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `HAVING` filters groups directly.

```python
def big_departments():
    return run_sql("""
        SELECT dept_id, AVG(salary) FROM employees
        WHERE dept_id IS NOT NULL
        GROUP BY dept_id
        HAVING COUNT(*) >= 2 AND AVG(salary) > 80
        ORDER BY dept_id""")
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Group in a subquery, then filter outside with WHERE. That costs O(n) time and O(d) space."

"I can do better: `HAVING` filters groups directly. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/groups-above-a-limit)

<a id="q4"></a>
### Q4. Inner Join

🟢 Warm-up · *JOIN*

Return each employee's name and department name. Employees without a department are left out. Order by employee id.

**Examples**

- `` → `[['Asha', 'Eng'], ['Ben', 'Eng'], ['Chen', 'Eng'], ['Dara', 'Eng'], ['Eli', 'Sales'], ['Fay', 'Sales'], ['Gus', 'Sales'], ['Hana', 'HR']]`

In [ ]:
def with_department():
    """
    Return each employee's name and department name. Employees without a
    department are left out. Order by employee id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(with_department, [
    ((), [['Asha', 'Eng'], ['Ben', 'Eng'], ['Chen', 'Eng'], ['Dara', 'Eng'], ['Eli', 'Sales'], ['Fay', 'Sales'], ['Gus', 'Sales'], ['Hana', 'HR']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A subquery in SELECT, then drop the missing ones.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`JOIN ... ON`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A subquery in SELECT, then drop the missing ones.

```python
def with_department():
    return run_sql("""
        SELECT name, dept FROM (
            SELECT id, name, (SELECT d.name FROM departments d WHERE d.id = e.dept_id) AS dept
            FROM employees e)
        WHERE dept IS NOT NULL
        ORDER BY id""")
```

**Cost:** O(n·d) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `JOIN ... ON`.

```python
def with_department():
    return run_sql("""
        SELECT e.name, d.name FROM employees e
        JOIN departments d ON e.dept_id = d.id
        ORDER BY e.id""")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A subquery in SELECT, then drop the missing ones. That costs O(n·d) time and O(n) space."

"I can do better: `JOIN ... ON`. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/inner-join)

<a id="q5"></a>
### Q5. Rows With No Match

🟢 Warm-up · *LEFT JOIN ... IS NULL*

Return names of departments with no employees, and names of customers with no orders, as two lists in a tuple.

**Examples**

- `` → `(['Legal'], ['Dev'])`

In [ ]:
def no_match():
    """
    Return names of departments with no employees, and names of customers with
    no orders, as two lists in a tuple.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(no_match, [
    ((), (['Legal'], ['Dev'])),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

`NOT IN` with a subquery. Beware: NOT IN returns nothing if the subquery contains a NULL.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`LEFT JOIN`, keeping rows where the right side is NULL. It handles NULLs safely.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** `NOT IN` with a subquery. Beware: NOT IN returns nothing if the subquery contains a NULL.

```python
def no_match():
    depts = run_sql("""SELECT name FROM departments
                       WHERE id NOT IN (SELECT dept_id FROM employees WHERE dept_id IS NOT NULL)""")
    custs = run_sql("SELECT name FROM customers WHERE id NOT IN (SELECT customer_id FROM orders)")
    return ([r[0] for r in depts], [r[0] for r in custs])
```

**Cost:** O(n·m) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `LEFT JOIN`, keeping rows where the right side is NULL. It handles NULLs safely.

```python
def no_match():
    depts = run_sql("""SELECT d.name FROM departments d
                       LEFT JOIN employees e ON e.dept_id = d.id WHERE e.id IS NULL""")
    custs = run_sql("""SELECT c.name FROM customers c
                       LEFT JOIN orders o ON o.customer_id = c.id WHERE o.id IS NULL""")
    return ([r[0] for r in depts], [r[0] for r in custs])
```

**Cost:** O(n + m) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: `NOT IN` with a subquery. Beware: NOT IN returns nothing if the subquery contains a NULL. That costs O(n·m) time and O(n) space."

"I can do better: `LEFT JOIN`, keeping rows where the right side is NULL. It handles NULLs safely. That's O(n + m) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/rows-with-no-match)

<a id="q6"></a>
### Q6. Find Duplicates

🟢 Warm-up · *GROUP BY + HAVING*

Return email addresses appearing more than once, sorted.

**Examples**

- `` → `['a@x.com', 'b@x.com']`

In [ ]:
def duplicate_emails():
    """
    Return email addresses appearing more than once, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(duplicate_emails, [
    ((), ['a@x.com', 'b@x.com']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Self join on the same email with a different id.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Group and keep counts above 1.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Self join on the same email with a different id.

```python
def duplicate_emails():
    rows = run_sql("""
        SELECT DISTINCT a.email FROM emails a
        JOIN emails b ON a.email = b.email AND a.id <> b.id
        ORDER BY a.email""")
    return [r[0] for r in rows]
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Group and keep counts above 1.

```python
def duplicate_emails():
    rows = run_sql("SELECT email FROM emails GROUP BY email HAVING COUNT(*) > 1 ORDER BY email")
    return [r[0] for r in rows]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Self join on the same email with a different id. That costs O(n²) time and O(n) space."

"I can do better: Group and keep counts above 1. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/find-duplicates)

<a id="q7"></a>
### Q7. COUNT and NULL

🟢 Warm-up · *A favourite trick question*

Return COUNT(\*), COUNT(dept\_id) and COUNT(DISTINCT dept\_id) from employees, as one row.

**Examples**

- `` → `[9, 8, 3]`

In [ ]:
def count_kinds():
    """
    Return COUNT(*), COUNT(dept_id) and COUNT(DISTINCT dept_id) from employees,
    as one row.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(count_kinds, [
    ((), [9, 8, 3]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fetch the column and count in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

All three in one query. Only `COUNT(*)` includes NULLs.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fetch the column and count in Python.

```python
def count_kinds():
    values = [r[0] for r in run_sql("SELECT dept_id FROM employees")]
    known = [v for v in values if v is not None]
    return [len(values), len(known), len(set(known))]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** All three in one query. Only `COUNT(*)` includes NULLs.

```python
def count_kinds():
    return run_sql("SELECT COUNT(*), COUNT(dept_id), COUNT(DISTINCT dept_id) FROM employees")[0]
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fetch the column and count in Python. That costs O(n) time and O(n) space."

"I can do better: All three in one query. Only `COUNT(*)` includes NULLs. That's O(n) time and O(d) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/count-and-null)

<a id="q8"></a>
### Q8. Employee and Their Manager

🟡 Interview standard · *Self join*

Return names of employees who earn more than their manager, ordered by name.

**Examples**

- `` → `[['Fay']]`

In [ ]:
def out_earn_manager():
    """
    Return names of employees who earn more than their manager, ordered by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(out_earn_manager, [
    ((), [['Fay']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A correlated subquery looking up the manager's salary per row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Join the table to itself, with two aliases.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A correlated subquery looking up the manager's salary per row.

```python
def out_earn_manager():
    return run_sql("""
        SELECT name FROM employees e
        WHERE salary > (SELECT m.salary FROM employees m WHERE m.id = e.manager_id)
        ORDER BY name""")
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Join the table to itself, with two aliases.

```python
def out_earn_manager():
    return run_sql("""
        SELECT e.name FROM employees e
        JOIN employees m ON e.manager_id = m.id
        WHERE e.salary > m.salary
        ORDER BY e.name""")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A correlated subquery looking up the manager's salary per row. That costs O(n²) time and O(n) space."

"I can do better: Join the table to itself, with two aliases. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/employee-and-their-manager)

<a id="q9"></a>
### Q9. Second Highest Salary

🟡 Interview standard · *The classic*

Return the second highest distinct salary.

**Examples**

- `` → `120`

In [ ]:
def second_highest():
    """
    Return the second highest distinct salary.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(second_highest, [
    ((), 120),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

MAX of everything below the MAX.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`DISTINCT`, `ORDER BY`, `LIMIT 1 OFFSET 1`. Change the offset for any N.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** MAX of everything below the MAX.

```python
def second_highest():
    return run_sql("""
        SELECT MAX(salary) FROM employees
        WHERE salary < (SELECT MAX(salary) FROM employees)""")[0][0]
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `DISTINCT`, `ORDER BY`, `LIMIT 1 OFFSET 1`. Change the offset for any N.

```python
def second_highest():
    return run_sql("""
        SELECT DISTINCT salary FROM employees
        ORDER BY salary DESC LIMIT 1 OFFSET 1""")[0][0]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: MAX of everything below the MAX. That costs O(n) time and O(1) space."

"I can do better: `DISTINCT`, `ORDER BY`, `LIMIT 1 OFFSET 1`. Change the offset for any N. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/second-highest-salary)

<a id="q10"></a>
### Q10. Keep the First of Each Duplicate

🟡 Interview standard · *De-duplicating*

Return (id, email) of the row with the lowest id for each email, ordered by id.

**Examples**

- `` → `[[1, 'a@x.com'], [2, 'b@x.com'], [4, 'c@x.com']]`

In [ ]:
def first_of_each():
    """
    Return (id, email) of the row with the lowest id for each email, ordered by
    id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(first_of_each, [
    ((), [[1, 'a@x.com'], [2, 'b@x.com'], [4, 'c@x.com']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

`NOT EXISTS` a lower id with the same email.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`ROW_NUMBER()` per email, keeping number 1. It also handles 'latest per user'.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** `NOT EXISTS` a lower id with the same email.

```python
def first_of_each():
    return run_sql("""
        SELECT id, email FROM emails a
        WHERE NOT EXISTS (SELECT 1 FROM emails b WHERE b.email = a.email AND b.id < a.id)
        ORDER BY id""")
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `ROW_NUMBER()` per email, keeping number 1. It also handles 'latest per user'.

```python
def first_of_each():
    return run_sql("""
        WITH ranked AS (
            SELECT id, email, ROW_NUMBER() OVER (PARTITION BY email ORDER BY id) AS rn
            FROM emails)
        SELECT id, email FROM ranked WHERE rn = 1 ORDER BY id""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: `NOT EXISTS` a lower id with the same email. That costs O(n²) time and O(n) space."

"I can do better: `ROW_NUMBER()` per email, keeping number 1. It also handles 'latest per user'. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/keep-the-first-of-each-duplicate)

<a id="q11"></a>
### Q11. Above the Department Average

🟡 Interview standard · *Correlated subquery vs window*

Return names of employees paid more than their own department's average, ordered by name.

**Examples**

- `` → `['Asha', 'Eli']`

In [ ]:
def above_dept_average():
    """
    Return names of employees paid more than their own department's average,
    ordered by name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(above_dept_average, [
    ((), ['Asha', 'Eli']),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A correlated subquery recomputes the average per row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Window `AVG(...) OVER (PARTITION BY ...)`, once.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A correlated subquery recomputes the average per row.

```python
def above_dept_average():
    rows = run_sql("""
        SELECT name FROM employees e
        WHERE salary > (SELECT AVG(salary) FROM employees x WHERE x.dept_id = e.dept_id)
        ORDER BY name""")
    return [r[0] for r in rows]
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Window `AVG(...) OVER (PARTITION BY ...)`, once.

```python
def above_dept_average():
    rows = run_sql("""
        WITH t AS (SELECT name, salary, AVG(salary) OVER (PARTITION BY dept_id) AS a
                   FROM employees WHERE dept_id IS NOT NULL)
        SELECT name FROM t WHERE salary > a ORDER BY name""")
    return [r[0] for r in rows]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A correlated subquery recomputes the average per row. That costs O(n²) time and O(1) space."

"I can do better: Window `AVG(...) OVER (PARTITION BY ...)`, once. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/above-the-department-average)

<a id="q12"></a>
### Q12. Rank Salaries

🟡 Interview standard · *ROW\_NUMBER, RANK, DENSE\_RANK*

For dept\_id 1, return name, ROW\_NUMBER, RANK and DENSE\_RANK by salary (highest first, ties by name).

**Examples**

- `` → `[['Asha', 1, 1, 1], ['Ben', 2, 2, 2], ['Chen', 3, 2, 2], ['Dara', 4, 4, 3]]`

In [ ]:
def rank_salaries():
    """
    For dept_id 1, return name, ROW_NUMBER, RANK and DENSE_RANK by salary
    (highest first, ties by name).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(rank_salaries, [
    ((), [['Asha', 1, 1, 1], ['Ben', 2, 2, 2], ['Chen', 3, 2, 2], ['Dara', 4, 4, 3]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Counting subqueries for each kind of rank.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Three window functions.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Counting subqueries for each kind of rank.

```python
def rank_salaries():
    return run_sql("""
        SELECT name,
          (SELECT COUNT(*) FROM employees b WHERE b.dept_id = 1
             AND (b.salary > a.salary OR (b.salary = a.salary AND b.name <= a.name))) AS rn,
          (SELECT COUNT(*) + 1 FROM employees b WHERE b.dept_id = 1 AND b.salary > a.salary) AS rk,
          (SELECT COUNT(DISTINCT salary) + 1 FROM employees b WHERE b.dept_id = 1 AND b.salary > a.salary) AS dr
        FROM employees a WHERE dept_id = 1 ORDER BY rn""")
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Three window functions.

```python
def rank_salaries():
    return run_sql("""
        SELECT name,
          ROW_NUMBER() OVER (ORDER BY salary DESC, name),
          RANK() OVER (ORDER BY salary DESC),
          DENSE_RANK() OVER (ORDER BY salary DESC)
        FROM employees WHERE dept_id = 1
        ORDER BY salary DESC, name""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Counting subqueries for each kind of rank. That costs O(n²) time and O(n) space."

"I can do better: Three window functions. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/rank-salaries)

<a id="q13"></a>
### Q13. Top Earner per Department

🟡 Interview standard · *Top N per group*

Return the department name and employee name of each department's highest paid person (all ties), ordered by department name, then employee.

**Examples**

- `` → `[['Eng', 'Asha'], ['HR', 'Hana'], ['Sales', 'Eli']]`

In [ ]:
def top_per_department():
    """
    Return the department name and employee name of each department's highest
    paid person (all ties), ordered by department name, then employee.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_per_department, [
    ((), [['Eng', 'Asha'], ['HR', 'Hana'], ['Sales', 'Eli']]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Join to a subquery holding each department's MAX.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`RANK()` per department, keeping rank 1. Change 1 to get top N.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Join to a subquery holding each department's MAX.

```python
def top_per_department():
    return run_sql("""
        SELECT d.name, e.name FROM employees e
        JOIN departments d ON d.id = e.dept_id
        JOIN (SELECT dept_id, MAX(salary) AS m FROM employees GROUP BY dept_id) t
          ON t.dept_id = e.dept_id AND t.m = e.salary
        ORDER BY d.name, e.name""")
```

**Cost:** O(n) time, O(d) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `RANK()` per department, keeping rank 1. Change 1 to get top N.

```python
def top_per_department():
    return run_sql("""
        WITH r AS (
            SELECT d.name AS dept, e.name AS emp,
                   RANK() OVER (PARTITION BY d.id ORDER BY e.salary DESC) AS rk
            FROM employees e JOIN departments d ON d.id = e.dept_id)
        SELECT dept, emp FROM r WHERE rk = 1 ORDER BY dept, emp""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Join to a subquery holding each department's MAX. That costs O(n) time and O(d) space."

"I can do better: `RANK()` per department, keeping rank 1. Change 1 to get top N. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/top-earner-per-department)

<a id="q14"></a>
### Q14. Running Total

🟡 Interview standard · *SUM OVER*

Return order id, amount and the running total of amount across all orders by date.

**Examples**

- `` → `[[1, 100, 100], [2, 50, 150], [3, 200, 350], [4, 70, 420], [5, 30, 450], [6, 80, 530], [7, 80, 610]]`

In [ ]:
def running_total():
    """
    Return order id, amount and the running total of amount across all orders by
    date.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(running_total, [
    ((), [[1, 100, 100], [2, 50, 150], [3, 200, 350], [4, 70, 420], [5, 30, 450], [6, 80, 530], [7, 80, 610]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Subquery summing all earlier rows, for each row.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`SUM(amount) OVER (ORDER BY ...)`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Subquery summing all earlier rows, for each row.

```python
def running_total():
    return run_sql("""
        SELECT id, amount,
          (SELECT SUM(amount) FROM orders b WHERE b.order_date <= a.order_date)
        FROM orders a ORDER BY order_date""")
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `SUM(amount) OVER (ORDER BY ...)`.

```python
def running_total():
    return run_sql("""
        SELECT id, amount, SUM(amount) OVER (ORDER BY order_date)
        FROM orders ORDER BY order_date""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Subquery summing all earlier rows, for each row. That costs O(n²) time and O(1) space."

"I can do better: `SUM(amount) OVER (ORDER BY ...)`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/running-total)

<a id="q15"></a>
### Q15. Share of Total

🟡 Interview standard · *Percent of the whole*

For each customer with orders, return name and their share of total revenue as a percentage rounded to 1 decimal, biggest first.

**Examples**

- `` → `[['Bob', 59.0], ['Ann', 36.1], ['Cid', 4.9]]`

In [ ]:
def revenue_share():
    """
    For each customer with orders, return name and their share of total revenue
    as a percentage rounded to 1 decimal, biggest first.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(revenue_share, [
    ((), [['Bob', 59.0], ['Ann', 36.1], ['Cid', 4.9]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

A scalar subquery for the grand total.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A window over the grouped totals: `SUM(SUM(amount)) OVER ()`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** A scalar subquery for the grand total.

```python
def revenue_share():
    return run_sql("""
        SELECT c.name, ROUND(100.0 * SUM(o.amount) / (SELECT SUM(amount) FROM orders), 1) AS pct
        FROM orders o JOIN customers c ON c.id = o.customer_id
        GROUP BY c.id ORDER BY pct DESC""")
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A window over the grouped totals: `SUM(SUM(amount)) OVER ()`.

```python
def revenue_share():
    return run_sql("""
        SELECT c.name, ROUND(100.0 * SUM(o.amount) / SUM(SUM(o.amount)) OVER (), 1) AS pct
        FROM orders o JOIN customers c ON c.id = o.customer_id
        GROUP BY c.id ORDER BY pct DESC""")
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: A scalar subquery for the grand total. That costs O(n) time and O(c) space."

"I can do better: A window over the grouped totals: `SUM(SUM(amount)) OVER ()`. That's O(n) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/share-of-total)

<a id="q16"></a>
### Q16. Pivot With CASE

🟡 Interview standard · *Rows to columns*

For each customer id with orders, return id and revenue in January, February and March as separate columns, ordered by id.

**Examples**

- `` → `[[1, 150, 70, 0], [2, 200, 0, 160], [3, 0, 30, 0]]`

In [ ]:
def pivot_months():
    """
    For each customer id with orders, return id and revenue in January, February
    and March as separate columns, ordered by id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(pivot_months, [
    ((), [[1, 150, 70, 0], [2, 200, 0, 160], [3, 0, 30, 0]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Three scalar subqueries per customer.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Conditional aggregation: `SUM(CASE WHEN ...)`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Three scalar subqueries per customer.

```python
def pivot_months():
    return run_sql("""
        SELECT id,
          (SELECT COALESCE(SUM(amount), 0) FROM orders WHERE customer_id = c.id AND order_date LIKE '2024-01%'),
          (SELECT COALESCE(SUM(amount), 0) FROM orders WHERE customer_id = c.id AND order_date LIKE '2024-02%'),
          (SELECT COALESCE(SUM(amount), 0) FROM orders WHERE customer_id = c.id AND order_date LIKE '2024-03%')
        FROM customers c WHERE id IN (SELECT customer_id FROM orders) ORDER BY id""")
```

**Cost:** O(c·n) time, O(c) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Conditional aggregation: `SUM(CASE WHEN ...)`.

```python
def pivot_months():
    return run_sql("""
        SELECT customer_id,
          SUM(CASE WHEN order_date LIKE '2024-01%' THEN amount ELSE 0 END),
          SUM(CASE WHEN order_date LIKE '2024-02%' THEN amount ELSE 0 END),
          SUM(CASE WHEN order_date LIKE '2024-03%' THEN amount ELSE 0 END)
        FROM orders GROUP BY customer_id ORDER BY customer_id""")
```

**Cost:** O(n) time, O(c) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Three scalar subqueries per customer. That costs O(c·n) time and O(c) space."

"I can do better: Conditional aggregation: `SUM(CASE WHEN ...)`. That's O(n) time and O(c) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/pivot-with-case)

<a id="q17"></a>
### Q17. Top 2 Distinct Salaries per Department

🔴 Stretch · *DENSE\_RANK*

Return department id, employee name and salary for everyone in their department's top 2 distinct salaries. Order by dept\_id, salary descending, name.

**Examples**

- `` → `[[1, 'Asha', 150], [1, 'Ben', 120], [1, 'Chen', 120], [2, 'Eli', 100], [2, 'Gus', 80], [3, 'Hana', 60]]`

In [ ]:
def top_two_salaries():
    """
    Return department id, employee name and salary for everyone in their
    department's top 2 distinct salaries. Order by dept_id, salary descending,
    name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(top_two_salaries, [
    ((), [[1, 'Asha', 150], [1, 'Ben', 120], [1, 'Chen', 120], [2, 'Eli', 100], [2, 'Gus', 80], [3, 'Hana', 60]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Count how many distinct higher salaries exist in the department.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`DENSE_RANK() <= 2`.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Count how many distinct higher salaries exist in the department.

```python
def top_two_salaries():
    return run_sql("""
        SELECT dept_id, name, salary FROM employees a
        WHERE dept_id IS NOT NULL AND
          (SELECT COUNT(DISTINCT salary) FROM employees b
           WHERE b.dept_id = a.dept_id AND b.salary > a.salary) < 2
        ORDER BY dept_id, salary DESC, name""")
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `DENSE_RANK() <= 2`.

```python
def top_two_salaries():
    return run_sql("""
        WITH r AS (
            SELECT dept_id, name, salary,
                   DENSE_RANK() OVER (PARTITION BY dept_id ORDER BY salary DESC) AS dr
            FROM employees WHERE dept_id IS NOT NULL)
        SELECT dept_id, name, salary FROM r WHERE dr <= 2
        ORDER BY dept_id, salary DESC, name""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Count how many distinct higher salaries exist in the department. That costs O(n²) time and O(n) space."

"I can do better: `DENSE_RANK() <= 2`. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/top-2-distinct-salaries-per-department)

<a id="q18"></a>
### Q18. Month-Over-Month Change

🔴 Stretch · *LAG*

For each month, return the month (YYYY-MM), revenue, and the change from the previous month (NULL for the first).

**Examples**

- `` → `[['2024-01', 350, None], ['2024-02', 100, -250], ['2024-03', 160, 60]]`

In [ ]:
def month_change():
    """
    For each month, return the month (YYYY-MM), revenue, and the change from the
    previous month (NULL for the first).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(month_change, [
    ((), [['2024-01', 350, None], ['2024-02', 100, -250], ['2024-03', 160, 60]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Self join each month's totals to the previous row in a CTE.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

`LAG()` reads the previous row.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Self join each month's totals to the previous row in a CTE.

```python
def month_change():
    return run_sql("""
        WITH m AS (SELECT substr(order_date, 1, 7) AS mon, SUM(amount) AS rev FROM orders GROUP BY mon)
        SELECT a.mon, a.rev,
          a.rev - (SELECT b.rev FROM m b WHERE b.mon < a.mon ORDER BY b.mon DESC LIMIT 1)
        FROM m a ORDER BY a.mon""")
```

**Cost:** O(m²) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** `LAG()` reads the previous row.

```python
def month_change():
    return run_sql("""
        WITH m AS (SELECT substr(order_date, 1, 7) AS mon, SUM(amount) AS rev FROM orders GROUP BY mon)
        SELECT mon, rev, rev - LAG(rev) OVER (ORDER BY mon) FROM m ORDER BY mon""")
```

**Cost:** O(m log m) time, O(m) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Self join each month's totals to the previous row in a CTE. That costs O(m²) time and O(m) space."

"I can do better: `LAG()` reads the previous row. That's O(m log m) time and O(m) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/month-over-month-change)

<a id="q19"></a>
### Q19. Consecutive Login Streaks

🔴 Stretch · *Gaps and islands*

For each user, return user\_id and their longest run of consecutive days, ordered by user\_id.

**Examples**

- `` → `[[1, 3], [2, 2], [3, 1]]`

In [ ]:
def longest_streak():
    """
    For each user, return user_id and their longest run of consecutive days,
    ordered by user_id.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(longest_streak, [
    ((), [[1, 3], [2, 2], [3, 1]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fetch rows and walk them in Python, tracking the current run.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Day minus ROW\_NUMBER is constant within a streak, so group by that difference.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fetch rows and walk them in Python, tracking the current run.

```python
def longest_streak():
    rows = run_sql("SELECT user_id, day FROM logins ORDER BY user_id, day")
    best = {}
    prev_user, prev_day, run = None, None, 0
    for user, day in rows:
        run = run + 1 if user == prev_user and day == prev_day + 1 else 1
        best[user] = max(best.get(user, 0), run)
        prev_user, prev_day = user, day
    return [[u, best[u]] for u in sorted(best)]
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** Day minus ROW\_NUMBER is constant within a streak, so group by that difference.

```python
def longest_streak():
    return run_sql("""
        WITH g AS (
            SELECT user_id, day - ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY day) AS grp
            FROM logins),
        s AS (SELECT user_id, COUNT(*) AS n FROM g GROUP BY user_id, grp)
        SELECT user_id, MAX(n) FROM s GROUP BY user_id ORDER BY user_id""")
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fetch rows and walk them in Python, tracking the current run. That costs O(n log n) time and O(n) space."

"I can do better: Day minus ROW\_NUMBER is constant within a streak, so group by that difference. That's O(n log n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/consecutive-login-streaks)

<a id="q20"></a>
### Q20. Walk the Org Chart

🔴 Stretch · *Recursive CTE*

Return every employee's name and depth below the top (Asha = 0), ordered by depth, then name.

**Examples**

- `` → `[['Asha', 0], ['Ben', 1], ['Chen', 1], ['Eli', 1], ['Hana', 1], ['Ivan', 1], ['Dara', 2], ['Fay', 2], ['Gus', 2]]`

In [ ]:
def org_depth():
    """
    Return every employee's name and depth below the top (Asha = 0), ordered by
    depth, then name.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(org_depth, [
    ((), [['Asha', 0], ['Ben', 1], ['Chen', 1], ['Eli', 1], ['Hana', 1], ['Ivan', 1], ['Dara', 2], ['Fay', 2], ['Gus', 2]]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Fetch the table and search it level by level in Python.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

A recursive CTE starts at the root and joins down, one level at a time.

</details>

<details>
<summary><b>✅ Answer: Step by step</b></summary>

**Idea:** Fetch the table and search it level by level in Python.

```python
def org_depth():
    rows = run_sql("SELECT id, name, manager_id FROM employees")
    depth = {}
    level, frontier = 0, {r[0] for r in rows if r[2] is None}
    while frontier:
        for i in frontier:
            depth[i] = level
        frontier = {r[0] for r in rows if r[2] in frontier}
        level += 1
    return sorted(([r[1], depth[r[0]]] for r in rows), key=lambda x: (x[1], x[0]))
```

**Cost:** O(n·d) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: The efficient way</b></summary>

**Idea:** A recursive CTE starts at the root and joins down, one level at a time.

```python
def org_depth():
    return run_sql("""
        WITH RECURSIVE tree(id, name, depth) AS (
            SELECT id, name, 0 FROM employees WHERE manager_id IS NULL
            UNION ALL
            SELECT e.id, e.name, t.depth + 1
            FROM employees e JOIN tree t ON e.manager_id = t.id)
        SELECT name, depth FROM tree ORDER BY depth, name""")
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Fetch the table and search it level by level in Python. That costs O(n·d) time and O(n) space."

"I can do better: A recursive CTE starts at the root and joins down, one level at a time. That's O(n) time and O(n) space."

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/interview/#/q/sql/walk-the-org-chart)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. WHERE vs HAVING?</b></summary>

WHERE filters rows before grouping. HAVING filters groups after grouping, so it can use aggregates.

</details>

<details>
<summary><b>2. INNER vs LEFT JOIN?</b></summary>

INNER keeps only matches. LEFT keeps every left row, with NULLs where there's no match.

</details>

<details>
<summary><b>3. RANK vs DENSE\_RANK vs ROW\_NUMBER?</b></summary>

On ties, ROW\_NUMBER still gives different numbers, RANK gives the same number then leaves a gap, and DENSE\_RANK gives the same number with no gap.

</details>

<details>
<summary><b>4. UNION vs UNION ALL?</b></summary>

UNION removes duplicates, which needs extra work. UNION ALL keeps everything and is faster, so use it when duplicates can't occur or are fine.

</details>

<details>
<summary><b>5. DELETE vs TRUNCATE vs DROP?</b></summary>

DELETE removes chosen rows and can be filtered. TRUNCATE quickly removes all rows and keeps the table. DROP removes the table itself. Exact behaviour around rollback differs by database.

</details>

<details>
<summary><b>6. What is a CTE, and why use one?</b></summary>

A named temporary result introduced by WITH. It makes complex queries readable, can be recursive, and lets you filter on window functions.

</details>

<details>
<summary><b>7. Correlated subquery?</b></summary>

A subquery that refers to the outer row, so it conceptually runs once per row. It can often be rewritten as a join or window function for better performance.

</details>

<details>
<summary><b>8. What is an index? What's the downside?</b></summary>

A sorted structure, often a B-tree, that speeds up lookups from O(n) to O(log n). It costs storage and slows writes.

</details>

<details>
<summary><b>9. Clustered vs non-clustered index?</b></summary>

A clustered index determines how rows are stored, so there's one per table. A non-clustered index is a separate structure that points to rows.

</details>

<details>
<summary><b>10. What does ACID mean?</b></summary>

Atomicity (all or nothing), consistency (rules hold), isolation (transactions don't interfere), durability (committed data survives failure).

</details>

<details>
<summary><b>11. What is normalisation?</b></summary>

Organising tables so each fact lives in one place, which avoids update anomalies. Analytics systems often denormalise to make reads faster.

</details>

<details>
<summary><b>12. How would you speed up a slow query?</b></summary>

Read the plan with EXPLAIN. Look for full scans and big joins. Add suitable indexes, filter early, select only needed columns, and avoid functions on indexed columns.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: How to read a query: the order SQL really runs in
- [ ] I can explain: Joins, and how they multiply rows
- [ ] I can explain: Subqueries and the second-highest salary
- [ ] I can explain: Window functions: the interview favourite
- [ ] I can explain: Hard patterns: gaps, islands, pivots and hierarchies
- [ ] I can explain: Indexes, performance and design

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) Pivot With CASE
- [ ] [Q17](#q17) Top 2 Distinct Salaries per Department
- [ ] [Q18](#q18) Month-Over-Month Change
- [ ] [Q19](#q19) Consecutive Login Streaks
- [ ] [Q20](#q20) Walk the Org Chart

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.